# 3D acceptance

An acceptance (efficiency) of $D^0 \to K^0_S \pi^+\pi^-$ events as a function of the
Dalitz-plot variables and of the decay time, as the `EfficiencyModel` of the C++ toy
generator in d02kspipi_toys. It is a product of components, each a function or a binned
model of some of the variables of `aman_examples.acceptance.acceptance_variables`:
$m^2(K^0_S\pi^\mp)$, $m^2(\pi^+\pi^-)$, $t/\tau$, and $\cos\theta_\mathrm{hel}$, the
helicity angle of the $\pi^+\pi^-$ system (between $K^0_S$ and $\pi^+$ in the
$\pi^+\pi^-$ frame).

Here the acceptance has a correlation between $m^2(\pi^+\pi^-)$,
$|\cos\theta_\mathrm{hel}|$ and $t$, binned as an efficiency map from simulation would
be. Toys with mixing are generated with it, and $x$, $y$ are fitted with the acceptance
in the normalisation, and without its Dalitz-plot part.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import uproot
from tensorwaves.data import IntensityDistributionGenerator
from tensorwaves.data.rng import NumpyUniformRNG
from tensorwaves.estimator import UnbinnedNLL
from tensorwaves.function.sympy import create_parametrized_function
from tensorwaves.optimizer import Minuit2

from aman_examples.acceptance import (
    FunctionAcceptance,
    HistogramAcceptance,
    acceptance_variables,
    apply_efficiency,
    decay_time_acceptance,
    efficiency_model_from_root,
)
from aman_examples.d02kspipi import formulate_model
from aman_examples.mixing import amplitude_transformer, formulate_mixing_model
from aman_examples.phasespace import DalitzDecayTimeGenerator

## Acceptance model

- a binned correlation model in $(m^2(\pi^+\pi^-), t/\tau, |\cos\theta_\mathrm{hel}|)$:
  $(1 - 0.25\,m^2(\pi^+\pi^-)\,e^{-t/2\tau}) \, (1 - 0.2\,|\cos\theta_\mathrm{hel}|^2)$:
  a loss at large $m^2(\pi^+\pi^-)$ that fades with the decay time, and one at large
  $|\cos\theta_\mathrm{hel}|$;
- a Dalitz-plot term $1 - 0.1\,m^2(K^0_S\pi^-)$;
- a decay-time turn-on.

In [ ]:
edges = [
    np.linspace(0.0, 3.0, 31),
    np.linspace(0.0, 10.0, 21),
    np.linspace(0.0, 1.0, 11),
]
m2_pipi, t, abs_cos = np.meshgrid(*((e[1:] + e[:-1]) / 2 for e in edges), indexing="ij")
correlation_map = (1 - 0.25 * m2_pipi * np.exp(-t / 2)) * (1 - 0.2 * abs_cos**2)
correlation = HistogramAcceptance(
    correlation_map, edges, ("m2_pipi", "t", "abs_cos_hel")
)
dalitz = FunctionAcceptance(lambda m2_kspim: 1 - 0.1 * m2_kspim, ("m2_kspim",))
time = decay_time_acceptance(t0=0.2, a=2.0, b=0.3, m=1.0, n=2.0)
acceptance = correlation * dalitz * time

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), tight_layout=True)
for ax, it in zip(axes, (0, 6), strict=True):
    mesh = ax.pcolormesh(
        edges[0], edges[2], correlation_map[:, it, :].T, vmin=0.2, vmax=1
    )
    ax.set_title(f"$t/\\tau$ in [{edges[1][it]:.1f}, {edges[1][it + 1]:.1f}]")
    ax.set_xlabel(r"$m^2(\pi^+\pi^-)$ [GeV$^2$]")
    ax.set_ylabel(r"$|\cos\theta_\mathrm{hel}|$")
fig.colorbar(mesh, ax=axes, label="correlation model")
plt.show()

## Toys with mixing and acceptance

As in the mixing example, with exaggerated $x = y = 0.05$.

In [ ]:
dalitz_model = formulate_model()
masses = [
    dalitz_model.parameter_defaults[dalitz_model.masses[k]]
    for k in ("pi", "KS", "pi", "D")
]
transformer = amplitude_transformer(dalitz_model)
mixing = formulate_mixing_model(x=0.05, y=0.05)
intensity = create_parametrized_function(
    mixing.intensity.doit(), mixing.parameter_defaults, backend="jax"
)
generator = DalitzDecayTimeGenerator(masses)
rng = NumpyUniformRNG(seed=0)
generated = IntensityDistributionGenerator(
    generator, intensity, domain_transformer=transformer, bunch_size=200_000
).generate(200_000, rng)
data = apply_efficiency(generated, acceptance(generated), rng=1)
print(f"{len(data['t'])} of {len(generated['t'])} events kept")

v_gen, v_acc = acceptance_variables(generated), acceptance_variables(data)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), tight_layout=True)
for ax, (name, label) in zip(
    axes,
    [
        ("m2_pipi", r"$m^2(\pi^+\pi^-)$ [GeV$^2$]"),
        ("abs_cos_hel", r"$|\cos\theta_\mathrm{hel}|$"),
        ("t", r"$t/\tau$"),
    ],
    strict=True,
):
    bins = np.linspace(0, 8, 41) if name == "t" else 40
    _, b, _ = ax.hist(
        v_gen[name], bins=bins, histtype="step", density=True, label="generated"
    )
    ax.hist(
        v_acc[name], bins=b, histtype="step", density=True, label="after acceptance"
    )
    ax.set_xlabel(label)
axes[0].legend()
plt.show()

## Fits

The normalisation sample is drawn uniformly in the Dalitz plot, with
exponential decay times, and the acceptance is applied to it as to the data. If only
the decay-time turn-on is taken into account, $y$ is biased: the acceptance changes the
Dalitz-plot distribution with the decay time, which the fit attributes to mixing.

In [ ]:
phsp = generator.generate(2_000_000, rng)
data_t = transformer(data)
results = {}
for label, model in (
    ("full 3D acceptance", acceptance),
    ("decay-time acceptance only", time),
):
    norm = apply_efficiency(phsp, model(phsp), rng=2)
    estimator = UnbinnedNLL(intensity, data_t, transformer(norm), backend="jax")
    results[label] = Minuit2().optimize(estimator, {"x": 0.0, "y": 0.0})
for label, result in results.items():
    values = ", ".join(
        f"{p} = {result.parameter_values[p]:.4f} ± {result.parameter_errors[p]:.4f}"
        f" (pull {(result.parameter_values[p] - 0.05) / result.parameter_errors[p]:+.1f})"
        for p in ("x", "y")
    )
    print(f"{label:28s} {values}")

## Acceptance from the ROOT files of d02kspipi_toys

`efficiency_model_from_root` builds the acceptance of `EfficiencyModel` from its ROOT
files: `"Dal"` (TH2D in $m^2(K^0_S\pi^+)$, $m^2(K^0_S\pi^-)$), `"DT"` (TH1D in
$t/\tau$, interpolated) and `"corr"` (TH3D in $m^2(\pi^+\pi^-)$, $t/\tau$,
$|\cos\theta_\mathrm{hel}|$, normalised to its maximum). Here the correlation map above
is written to a file and read back.

In [ ]:
with TemporaryDirectory() as directory:
    file_name = Path(directory) / "corr.root"
    with uproot.recreate(file_name) as f:
        f["corr"] = (correlation_map, *edges)
    from_root = efficiency_model_from_root(correlations=file_name, min_decay_time=0.0)
    print(
        np.allclose(
            from_root(generated), correlation(generated) / correlation_map.max()
        )
    )